# Derivação numérica e diferenciação automática

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/10-derivacao.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. deduzir fórmulas de diferenças finitas pela série de Taylor e prever a sua ordem;
> 2. escolher o passo $h$ equilibrando o erro de truncamento com o arredondamento ou o ruído dos dados;
> 3. melhorar uma derivada com a extrapolação de Richardson;
> 4. distinguir derivação simbólica, numérica e automática, e usar `torch.autograd`;
> 5. reconhecer quando derivar dados medidos amplifica o ruído, e o que fazer.

Este capítulo usa PyTorch na seção 3.5 e em parte das seções 5 e 7. O PyTorch não roda no navegador: para essas células, use o Colab ou uma instalação local.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
from cnlivro.derivacao import (progressiva, regressiva, centrada, segunda_centrada, passo_complexo,
                               richardson, derivada_dados)

## 1. Problema de engenharia: o conforto de um elevador

Um elevador novo precisa passar por um ensaio de conforto antes de ser entregue. O contrato especifica que a **aceleração** da cabine não pode passar de $1{,}2\ \text{m/s}^2$ em módulo. Acima disso, os passageiros sentem o "frio na barriga" na partida e o "peso" na frenagem.

A equipe de comissionamento não tem um acelerômetro à mão, mas tem um **medidor de distância a laser** apontado para o teto da cabine. Ele registra a posição $s(t)$ cinquenta vezes por segundo ($\Delta t = 0{,}02$ s), com incerteza de cerca de $0{,}5$ mm em cada leitura. A aceleração é a **segunda derivada** da posição:

$$
a(t) = \frac{d^2 s}{dt^2}.
$$

O registro é uma tabela de números, não uma fórmula. Derivar uma tabela é o assunto deste capítulo, e a seção 6 mostra que esse problema, aparentemente simples, esconde uma armadilha.

Para que possamos comparar com a resposta certa, os dados abaixo são **simulados**: uma viagem de 13,5 m em 12 s, com aceleração máxima de exatamente $1\ \text{m/s}^2$, mais o ruído de $0{,}5$ mm do sensor.

In [ ]:
# Dados simulados de uma viagem do elevador: partida suave (3 s), velocidade constante (6 s), frenagem (3 s)
A_MAX, T_RAMPA, T_CRUZEIRO = 1.0, 3.0, 6.0          # m/s², s, s
V_CRUZEIRO = A_MAX * T_RAMPA / 2                     # 1,5 m/s

def _rampa(tt):        # deslocamento durante a partida: a(t) = A sen²(πt/T)
    return A_MAX * (tt**2 / 4 + T_RAMPA**2 / (8 * np.pi**2) * (np.cos(2 * np.pi * tt / T_RAMPA) - 1))

def posicao_exata(t):
    t = np.asarray(t, dtype=float)
    t1, t2, t3 = T_RAMPA, T_RAMPA + T_CRUZEIRO, 2 * T_RAMPA + T_CRUZEIRO
    s1 = _rampa(T_RAMPA)
    return np.select([t < 0, t < t1, t < t2, t < t3],
                     [0 * t, _rampa(np.clip(t, 0, t1)), s1 + V_CRUZEIRO * (t - t1),
                      s1 + V_CRUZEIRO * T_CRUZEIRO + V_CRUZEIRO * (t - t2) - _rampa(np.clip(t - t2, 0, T_RAMPA))],
                     2 * s1 + V_CRUZEIRO * T_CRUZEIRO)

def aceleracao_exata(t):
    t = np.asarray(t, dtype=float)
    subida = (t >= 0) & (t < T_RAMPA)
    descida = (t >= T_RAMPA + T_CRUZEIRO) & (t < 2 * T_RAMPA + T_CRUZEIRO)
    return (A_MAX * np.sin(np.pi * t / T_RAMPA) ** 2 * subida
            - A_MAX * np.sin(np.pi * (t - T_RAMPA - T_CRUZEIRO) / T_RAMPA) ** 2 * descida)

dt = 0.02                                            # 50 leituras por segundo
t = np.arange(-1.0, 14.0, dt)
sigma = 0.5e-3                                       # incerteza do laser: 0,5 mm
s = posicao_exata(t) + np.random.default_rng(7).normal(0.0, sigma, t.size)
print(f"{t.size} leituras; deslocamento total = {s[-1] - s[0]:.3f} m")

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["registro do laser (posição)", "zoom: 0,2 s no meio da viagem"],
                    horizontal_spacing=0.1)
fig.add_trace(go.Scatter(x=t, y=s, mode="lines", name="posição medida", line=dict(color=CORES["aproximacao"])), 1, 1)
z = (t > 6.0) & (t < 6.2)
fig.add_trace(go.Scatter(x=t[z], y=1000 * s[z], mode="markers", name="leituras (mm)",
                         marker=dict(color=CORES["aproximacao"], size=8)), 1, 2)
fig.add_trace(go.Scatter(x=t[z], y=1000 * posicao_exata(t[z]), mode="lines", name="posição exata",
                         line=dict(color=CORES["exata"], dash="dot")), 1, 2)
fig.update_xaxes(title="tempo (s)")
fig.update_yaxes(title="posição (m)", row=1, col=1)
fig.update_yaxes(title="posição (mm)", row=1, col=2)
fig.update_layout(width=980, height=400, legend=dict(orientation="h", y=-0.2))
mostrar(fig)

**O que notar:** na escala da viagem, a curva é lisa. No detalhe, cada leitura se afasta da posição exata por uma fração de milímetro. É um sensor muito bom. Guarde essa imagem: o ruído é invisível na posição.

## 2. A ideia em uma imagem

A derivada $f'(x)$ é a inclinação da reta tangente. Sem a fórmula de $f$, só podemos calcular a inclinação de uma reta **secante**, que passa por dois pontos da curva. Arraste o slider para diminuir a distância $h$ entre os pontos.

In [ ]:
f_ex = lambda x: 1 + 0.6 * np.sin(2 * x)
df_ex = lambda x: 1.2 * np.cos(2 * x)
x0 = 0.5
xs = np.linspace(-0.2, 1.6, 300)

def quadro(h):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=xs, y=f_ex(xs), mode="lines", name="f(x)", line=dict(color=CORES["exata"], width=2)))
    reta = lambda m, xa, ya: ya + m * (xs - xa)
    fig.add_trace(go.Scatter(x=xs, y=reta(df_ex(x0), x0, f_ex(x0)), mode="lines", name="tangente (exata)",
                             line=dict(color=CORES["referencia"], dash="dot")))
    m_prog = (f_ex(x0 + h) - f_ex(x0)) / h
    m_cent = (f_ex(x0 + h) - f_ex(x0 - h)) / (2 * h)
    fig.add_trace(go.Scatter(x=xs, y=reta(m_prog, x0, f_ex(x0)), mode="lines",
                             name=f"progressiva: {numero(m_prog, 3)}", line=dict(color=CORES["erro"])))
    fig.add_trace(go.Scatter(x=xs, y=reta(m_cent, x0 - h, f_ex(x0 - h)), mode="lines",
                             name=f"centrada: {numero(m_cent, 3)}", line=dict(color=CORES["aproximacao"])))
    fig.add_trace(go.Scatter(x=[x0 - h, x0, x0 + h], y=f_ex(np.array([x0 - h, x0, x0 + h])), mode="markers",
                             marker=dict(color=CORES["destaque"], size=10), name="pontos usados"))
    fig.update_xaxes(title="x", range=[-0.2, 1.6])
    fig.update_yaxes(title="f(x)", range=[0.2, 2.1])
    fig.update_layout(width=760, height=500, legend=dict(orientation="h", y=-0.2),
                      title=f"h = {numero(h)}: derivada exata {numero(df_ex(x0), 3)}")
    return fig

hs_slider = [0.5, 0.3, 0.2, 0.1, 0.05]
quadros = [(numero(h), quadro(h)) for h in hs_slider]
mostrar(com_controle(quadros, rotulo="h = ", ativo=0, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** a secante **progressiva** (vermelha), que usa $x_0$ e $x_0 + h$, fica visivelmente torta até $h$ bem pequeno. A secante **centrada** (azul), que usa $x_0 - h$ e $x_0 + h$, é quase paralela à tangente já com $h = 0{,}3$. A mesma quantidade de avaliações de $f$, posicionada de forma simétrica, dá uma resposta muito melhor. A seção 3 explica por quê.

## 3. Formulação matemática

### 3.1 Diferenças finitas a partir de Taylor

A série de Taylor de $f$ em torno de $x$ é

$$
f(x + h) = f(x) + h f'(x) + \frac{h^2}{2} f''(x) + \frac{h^3}{6} f'''(x) + \cdots
$$ (eq-taylor-mais)

Isolando $f'(x)$:

$$
f'(x) = \underbrace{\frac{f(x + h) - f(x)}{h}}_{\text{diferença progressiva}} - \frac{h}{2} f''(\xi).
$$

O erro de **truncamento** é proporcional a $h$: a fórmula é de **primeira ordem**. Escrevendo também

$$
f(x - h) = f(x) - h f'(x) + \frac{h^2}{2} f''(x) - \frac{h^3}{6} f'''(x) + \cdots
$$ (eq-taylor-menos)

e subtraindo {eq}`eq-taylor-menos` de {eq}`eq-taylor-mais`, os termos com $h^2$ se cancelam:

$$
f'(x) = \underbrace{\frac{f(x + h) - f(x - h)}{2h}}_{\text{diferença centrada}} - \frac{h^2}{6} f'''(\xi).
$$ (eq-centrada)

A diferença centrada é de **segunda ordem**: dividir $h$ por 2 divide o erro por 4. Somando as duas séries, em vez de subtrair, saem os termos ímpares, e obtemos a segunda derivada:

$$
f''(x) = \frac{f(x + h) - 2f(x) + f(x - h)}{h^2} - \frac{h^2}{12} f^{(4)}(\xi).
$$ (eq-segunda)

Combinando mais pontos, saem fórmulas de ordem mais alta e fórmulas **unilaterais**, necessárias nas pontas de uma tabela, onde não há vizinho de um dos lados. A mais usada é a de três pontos, de segunda ordem:

$$
f'(x_0) \approx \frac{-3f(x_0) + 4f(x_0 + h) - f(x_0 + 2h)}{2h}.
$$

Para dados com espaçamento **irregular**, deriva-se a parábola que passa pelos três pontos vizinhos (capítulo 6). É o que fazem `np.gradient(y, x, edge_order=2)` e a função de referência `derivada_dados`.

### 3.2 O erro total: truncamento contra arredondamento

O computador não calcula $f(x)$ exatamente, mas $f(x)(1 + \delta)$, com $|\delta| \le \epsilon \approx 10^{-16}$ (capítulo 2). Na diferença progressiva, esses erros se somam e são **divididos por $h$**:

$$
E(h) \le \underbrace{\frac{h}{2}|f''|}_{\text{truncamento}} + \underbrace{\frac{2\epsilon |f|}{h}}_{\text{arredondamento}} .
$$

Um termo cai e o outro cresce quando $h$ diminui. O mínimo fica em $h^* \approx 2\sqrt{\epsilon |f|/|f''|} \sim 10^{-8}$, com erro $\sim 10^{-8}$: **metade dos algarismos se perde**. Para a centrada, $E(h) \le \frac{h^2}{6}|f'''| + \frac{\epsilon|f|}{h}$, com $h^* \sim \epsilon^{1/3} \approx 10^{-5}$ e erro $\sim 10^{-11}$.

Com **dados medidos**, o papel de $\epsilon|f|$ é feito pela incerteza $\sigma$ de cada leitura, que é muitas ordens de grandeza maior. Para a segunda derivada {eq}`eq-segunda`, o ruído é dividido por $h^2$:

$$
E(h) \le \frac{h^2}{12} M_4 + \frac{4\sigma}{h^2}, \qquad h^* = \left(\frac{48\,\sigma}{M_4}\right)^{1/4},
$$ (eq-passo-otimo)

com $M_4$ uma estimativa de $|f^{(4)}|$. Se os erros das leituras são independentes, o desvio padrão do ruído na segunda derivada é $\sqrt{6}\,\sigma/h^2$. **Diminuir o passo não é sempre melhor.**

### 3.3 Extrapolação de Richardson

Se $D(h)$ é a diferença centrada, o erro tem a forma $D(h) = f'(x) + c_2 h^2 + c_4 h^4 + \cdots$. Calculando com $h$ e $h/2$:

$$
D(h/2) = f'(x) + c_2 \frac{h^2}{4} + \cdots
\quad\Longrightarrow\quad
f'(x) \approx \frac{4D(h/2) - D(h)}{3} + O(h^4).
$$ (eq-richardson)

A combinação elimina o termo $h^2$ sem nenhuma avaliação extra além das duas diferenças. Repetindo a ideia, cada coluna de uma tabela elimina mais um termo. A mesma ideia, aplicada à regra do trapézio, é a integração de Romberg (capítulo 11). Richardson só funciona se o erro tem de fato a forma prevista, o que exige $f$ suave e $h$ longe da região dominada pelo arredondamento.

### 3.4 O passo complexo

Para funções analíticas, há um truque que evita a subtração. Com um passo **imaginário**,

$$
f(x + ih) = f(x) + ih f'(x) - \frac{h^2}{2}f''(x) - \cdots
\quad\Longrightarrow\quad
f'(x) = \frac{\operatorname{Im} f(x + ih)}{h} + O(h^2).
$$

Não há diferença de números próximos, então $h$ pode ser $10^{-20}$, e a derivada sai com precisão de máquina. Exige que o código de $f$ aceite números complexos e não use `abs`, `max` ou comparações.

### 3.5 Três maneiras de derivar

| | Simbólica (SymPy) | Numérica (diferenças) | Automática (PyTorch) |
| --- | --- | --- | --- |
| Entrada | uma **fórmula** | **valores** de $f$ | um **programa** que calcula $f$ |
| Saída | outra fórmula | um número aproximado | o valor exato da derivada do programa |
| Erro | nenhum | truncamento + arredondamento | só arredondamento |
| Custo do gradiente de $f:\mathbb{R}^n \to \mathbb{R}$ | fórmulas que crescem muito | $n + 1$ avaliações de $f$ | cerca de 3 avaliações de $f$ |

A **diferenciação automática** aplica a regra da cadeia a cada operação elementar que o programa executa. Ela não aproxima nada e não manipula fórmulas: cada operação ($+$, $\times$, $\sin$...) sabe a sua própria derivada, e o programa propaga esses números.

No **modo direto** (*forward*), cada número carrega junto o valor da sua derivada. Um "número dual" $a + b\varepsilon$, com $\varepsilon^2 = 0$, faz isso: $(a + b\varepsilon)(c + d\varepsilon) = ac + (ad + bc)\varepsilon$ é exatamente a regra do produto. Bastam algumas linhas:

In [ ]:
class Dual:
    # Número dual a + b ε, com ε² = 0: 'valor' carrega f e 'deriv' carrega f'
    def __init__(self, valor, deriv=0.0):
        self.valor, self.deriv = valor, deriv
    def __add__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.valor + o.valor, self.deriv + o.deriv)
    __radd__ = __add__
    def __mul__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.valor * o.valor, self.valor * o.deriv + self.deriv * o.valor)   # regra do produto
    __rmul__ = __mul__
    def __truediv__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.valor / o.valor, (self.deriv * o.valor - self.valor * o.deriv) / o.valor**2)

def seno(u):      # regra da cadeia: (sen u)' = cos(u) · u'
    return Dual(np.sin(u.valor), np.cos(u.valor) * u.deriv)

def exp(u):
    return Dual(np.exp(u.valor), np.exp(u.valor) * u.deriv)

def g(x):         # um programa qualquer, escrito sem pensar em derivadas
    return x * seno(x) + exp(x) / x

x = Dual(1.5, 1.0)            # semente: dx/dx = 1
r = g(x)
exata = np.sin(1.5) + 1.5 * np.cos(1.5) + np.exp(1.5) * (1.5 - 1) / 1.5**2
print(f"g(1,5) = {r.valor:.12f};  g'(1,5) automática = {r.deriv:.15f};  exata = {exata:.15f}")

No **modo reverso** (*reverse*), o programa é executado uma vez guardando todas as operações num grafo, e depois as derivadas são propagadas **de trás para frente**, da saída para as entradas. Para uma função escalar de $n$ variáveis, como a perda de uma rede neural com milhões de pesos (capítulo 9), o modo reverso dá o gradiente inteiro por um custo de poucas avaliações de $f$, contra $n + 1$ avaliações das diferenças finitas. É a **retropropagação** (*backpropagation*), e é o que o PyTorch faz:

In [ ]:
import torch
torch.set_default_dtype(torch.float64)       # PyTorch usa float32 por padrão; aqui queremos 16 algarismos

x = torch.tensor(1.5, requires_grad=True)
y = x * torch.sin(x) + torch.exp(x) / x       # o mesmo programa, com operações do PyTorch
y.backward()                                  # modo reverso: preenche x.grad com dy/dx
print(f"g'(1,5) pelo torch.autograd = {x.grad.item():.15f}")

## 4. À mão

Vamos estimar $f'(2)$ para $f(x) = \ln x$, cuja derivada exata é $1/2 = 0{,}5$.

Com $h = 0{,}1$: $\ln 2{,}1 = 0{,}741937$, $\ln 2 = 0{,}693147$ e $\ln 1{,}9 = 0{,}641854$.

- **Progressiva:** $(0{,}741937 - 0{,}693147)/0{,}1 = 0{,}48790$. Erro de $0{,}012$.
- **Centrada:** $(0{,}741937 - 0{,}641854)/0{,}2 = 0{,}500417$. Erro de $0{,}0004$, trinta vezes menor, com o mesmo número de avaliações.

Com $h = 0{,}05$: $\ln 2{,}05 = 0{,}717840$ e $\ln 1{,}95 = 0{,}667829$, e a centrada dá $0{,}100011/0{,}1 = 0{,}500104$. O erro caiu de $0{,}000417$ para $0{,}000104$, um fator 4: segunda ordem, como previsto.

- **Richardson** {eq}`eq-richardson`: $(4 \times 0{,}500104 - 0{,}500417)/3 = 0{,}4999997$. Erro de $3 \times 10^{-7}$, mil vezes menor que o da melhor centrada, sem nenhuma avaliação nova de $f$.

(Com seis casas nas contas à mão, a última casa de Richardson é incerta; o computador, com todos os algarismos, dá $0{,}49999984$.)

In [ ]:
print(f"progressiva, h = 0,1 : {progressiva(np.log, 2.0, 0.1):.6f}")
print(f"centrada,    h = 0,1 : {centrada(np.log, 2.0, 0.1):.6f}")
print(f"centrada,    h = 0,05: {centrada(np.log, 2.0, 0.05):.6f}")
tab = richardson(np.log, 2.0, 0.1, niveis=3)
print("tabela de Richardson (linhas: h, h/2, h/4; colunas: extrapolações):")
print(np.array2string(tab.tabela, precision=10, floatmode="fixed"))

## 5. Explore

### 5.1 A curva em V do erro

Para $f(x) = \sin x$ em $x = 1$, a figura mostra o erro de cada fórmula em função do passo $h$, de $10^{-16}$ a $1$.

In [ ]:
hs = np.logspace(-16, 0, 161)
x0, exata = 1.0, np.cos(1.0)
erros = {
    "progressiva (ordem 1)": [abs(progressiva(np.sin, x0, h) - exata) for h in hs],
    "centrada (ordem 2)": [abs(centrada(np.sin, x0, h) - exata) for h in hs],
    "Richardson, 2 níveis (ordem 4)": [abs(richardson(np.sin, x0, h, niveis=2).valor - exata) for h in hs],
    "passo complexo": [abs(passo_complexo(np.sin, x0, h) - exata) for h in hs],
}
cores = [CORES["erro"], CORES["aproximacao"], CORES["secundaria"], CORES["destaque"]]
fig = go.Figure()
for (nome, e), cor in zip(erros.items(), cores):
    e = np.maximum(e, 1e-17)                       # erro zero não aparece em escala log
    fig.add_trace(go.Scatter(x=hs, y=e, mode="lines", name=nome, line=dict(color=cor, width=2)))
fig.add_hline(y=np.finfo(float).eps, line=dict(color=CORES["referencia"], dash="dot"),
              annotation_text="ε da máquina", annotation_position="bottom right")
fig.update_xaxes(type="log", title="passo h", exponentformat="power")
fig.update_yaxes(type="log", title="erro absoluto", exponentformat="power", range=[-17.5, 1])
fig.update_layout(width=820, height=480, legend=dict(x=0.02, y=0.02), title="Erro da derivada de sen x em x = 1")
mostrar(fig)

**O que notar:**
- À **direita**, cada curva desce com a inclinação da sua ordem: 1, 2 e 4. É o truncamento.
- À **esquerda**, as três fórmulas de diferenças sobem com inclinação $-1$: é o arredondamento dividido por $h$. O fundo do V fica em $h \approx 10^{-8}$ para a progressiva, com erro de cerca de $10^{-8}$, e em $h \approx 10^{-5}$ para a centrada, com erro de $10^{-11}$, como previsto na seção 3.2. Richardson melhora a parte direita, mas não a esquerda.
- O **passo complexo** não tem lado esquerdo: para $h \le 10^{-8}$, o erro fica no nível de $\epsilon$.

Um código que use `h = 1e-12` "para ser mais preciso" perde quase todos os algarismos.

### 5.2 Dados com ruído: o V se desloca

Medidas reais têm erros muito maiores que $\epsilon$. Arraste o slider para acrescentar um ruído de tamanho $\sigma$ aos valores de $\sin x$ e veja o que acontece com a curva de erro da diferença centrada.

In [ ]:
rng = np.random.default_rng(1)
hs = np.logspace(-8, 0, 81)

def quadro_ruido(sig):
    def f_ruidosa(x):
        return np.sin(x) + sig * rng.normal()        # cada leitura tem o seu erro
    e = [abs(centrada(f_ruidosa, 1.0, h) - np.cos(1.0)) for h in hs]
    teoria = hs**2 / 6 * np.cos(1.0) + (sig + 1e-16) / hs
    h_otimo = (3 * (sig + 1e-16) / np.cos(1.0)) ** (1 / 3)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hs, y=e, mode="markers", name="erro medido", marker=dict(color=CORES["erro"], size=6)))
    fig.add_trace(go.Scatter(x=hs, y=teoria, mode="lines", name="h²|f'''|/6 + σ/h",
                             line=dict(color=CORES["exata"], dash="dash")))
    # linha vertical como traço (e não add_vline), para mudar junto com o slider
    fig.add_trace(go.Scatter(x=[h_otimo, h_otimo], y=[1e-12, 1e5], mode="lines",
                             name=f"passo ótimo h* ≈ {h_otimo:.0e}".replace(".", ","),
                             line=dict(color=CORES["destaque"], dash="dot")))
    fig.update_xaxes(type="log", title="passo h", exponentformat="power")
    fig.update_yaxes(type="log", title="erro da derivada", exponentformat="power", range=[-12, 5])
    fig.update_layout(width=760, height=460, legend=dict(x=0.6, y=0.98),
                      title=f"σ = {sig:.0e}".replace(".", ",") if sig else "σ = 0 (só arredondamento)")
    return fig

sigmas = [0.0, 1e-10, 1e-7, 1e-4, 1e-2]
quadros = [((f"{s_:.0e}" if s_ else "0"), quadro_ruido(s_)) for s_ in sigmas]
mostrar(com_controle(quadros, rotulo="σ = ", ativo=3, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** o ruído entra no lugar de $\epsilon$. Com $\sigma = 10^{-4}$, um ruído na quarta casa decimal, o passo ótimo já é da ordem de $0{,}08$ e o melhor erro possível é de cerca de $10^{-3}$: a derivada tem só três algarismos, mesmo que os dados tenham quatro. Com $\sigma = 10^{-2}$, o melhor passo passa de $0{,}3$. **Quanto mais ruidosos os dados, maior precisa ser o passo**, e pior é a melhor derivada possível.

### 5.3 Diferenciação automática através de uma simulação

A diferenciação automática deriva **programas**, inclusive laços. No capítulo 1, a velocidade do paraquedista foi calculada pelo método de Euler. Quanto a velocidade em $t = 10$ s muda se o coeficiente de arrasto $c$ mudar um pouco? Essa **sensibilidade** $\partial v(10)/\partial c$ é o tipo de derivada que se usa para calibrar modelos e para otimizar projetos.

In [ ]:
g_, m_ = 9.81, 68.1
T_final, passo = 10.0, 0.5

def velocidade_euler(c):                     # o mesmo laço do capítulo 1
    v = 0.0 * c
    for _ in range(int(T_final / passo)):
        v = v + passo * (g_ - c / m_ * v)
    return v

c = torch.tensor(12.5, requires_grad=True)
v10 = velocidade_euler(c)
v10.backward()
dv_auto = c.grad.item()
dv_dif = centrada(lambda cc: velocidade_euler(cc), 12.5, 1e-4)
dv_exata = (-g_ * m_ / 12.5**2 * (1 - np.exp(-12.5 * T_final / m_))
            + g_ * T_final / 12.5 * np.exp(-12.5 * T_final / m_))
print(f"v(10 s) por Euler = {v10.item():.4f} m/s")
print(f"∂v/∂c automática (através do laço de Euler)   = {dv_auto:.6f}")
print(f"∂v/∂c por diferença centrada do mesmo código  = {dv_dif:.6f}")
print(f"∂v/∂c da solução exata da EDO                 = {dv_exata:.6f}")

**O que notar:** a derivada automática e a diferença centrada concordam em todos os algarismos mostrados: as duas derivam **o mesmo programa**. Mas as duas diferem da derivada da solução exata em cerca de 2 %. A diferenciação automática é exata **para o código**, não para a física: ela deriva a aproximação de Euler, com o seu passo de $0{,}5$ s. O erro de discretização continua lá, e diminui com o passo, como no capítulo 12.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Tenho o registro de posição de um elevador, medido por um laser a 50 Hz (vetores `t` em segundos e `s` em metros). Escreva uma função que calcule a velocidade e a aceleração com diferenças centradas de segunda ordem, e diga se a aceleração máxima em módulo passa do limite de 1,2 m/s² do contrato."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
def avalia_conforto(t, s, limite=1.2):
    h = t[1] - t[0]
    v = (s[2:] - s[:-2]) / (2 * h)                     # velocidade: diferença centrada, O(h²)
    a = (s[2:] - 2 * s[1:-1] + s[:-2]) / h**2          # aceleração: diferença centrada, O(h²)
    a_max = np.max(np.abs(a))
    print(f"velocidade máxima: {np.max(np.abs(v)):.2f} m/s")
    print(f"aceleração máxima: {a_max:.2f} m/s²")
    print("APROVADO" if a_max <= limite else "REPROVADO: aceleração acima do limite")
    return t[1:-1], v, a

t_ag, v_ag, a_ag = avalia_conforto(t, s)

As fórmulas estão certas, são de segunda ordem, e a velocidade máxima, $1{,}56$ m/s, é plausível. O elevador seria reprovado com uma aceleração de mais de $9\ \text{m/s}^2$, quase a da gravidade.

### 6.2 Verificação

**Plausibilidade física.** Uma aceleração de $9\ \text{m/s}^2$, mesmo durante $0{,}02$ s, mudaria a velocidade em $0{,}18$ m/s de uma leitura para a seguinte. Passageiros quase flutuariam. Vale olhar o resultado inteiro, e não só o máximo:

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=["velocidade calculada pelo agente", "aceleração calculada pelo agente"])
fig.add_trace(go.Scatter(x=t_ag, y=v_ag, mode="lines", name="velocidade (agente)",
                         line=dict(color=CORES["aproximacao"], width=1)), 1, 1)
fig.add_trace(go.Scatter(x=t_ag, y=a_ag, mode="lines", name="aceleração (agente)",
                         line=dict(color=CORES["erro"], width=1)), 2, 1)
fig.add_trace(go.Scatter(x=t, y=aceleracao_exata(t), mode="lines", name="aceleração exata",
                         line=dict(color=CORES["exata"], width=3)), 2, 1)
for y_lim in (1.2, -1.2):
    fig.add_hline(y=y_lim, line=dict(color=CORES["referencia"], dash="dash"), row=2, col=1)
fig.update_xaxes(title="tempo (s)", row=2, col=1)
fig.update_yaxes(title="v (m/s)", row=1, col=1)
fig.update_yaxes(title="a (m/s²)", row=2, col=1)
fig.update_layout(width=900, height=620, legend=dict(orientation="h", y=-0.12))
mostrar(fig)

A velocidade é razoável, com um chiado pequeno. A aceleração é **ruído puro**: a curva exata, em preto, some dentro de uma faixa de $\pm 9\ \text{m/s}^2$.

**Um trecho com resposta conhecida.** Entre $t = 4$ s e $t = 8$ s, o elevador anda com velocidade constante, e a aceleração verdadeira é **zero**. O que o código devolve ali é só erro, e a seção 3.2 prevê o seu tamanho, $\sqrt6\,\sigma/h^2$:

In [ ]:
cruzeiro = (t_ag > 4) & (t_ag < 8)
print(f"desvio padrão da aceleração no cruzeiro: {a_ag[cruzeiro].std():.2f} m/s²")
print(f"previsão √6·σ/h² com σ = 0,5 mm, h = 0,02 s: {np.sqrt(6) * sigma / dt**2:.2f} m/s²")
print(f"previsão para a velocidade, σ/(√2·h): {sigma / (np.sqrt(2) * dt):.3f} m/s")

A teoria explica o resultado: com $h = 0{,}02$ s, o fator $1/h^2 = 2500$ transforma meio milímetro de ruído em $3\ \text{m/s}^2$ de desvio padrão, e o máximo entre 750 leituras chega a três desvios padrão. A velocidade sofre bem menos, porque o ruído só é dividido por $h$.

**Um teste em que o código passa.** Com dados **sem ruído**, o mesmo código acerta, e a ordem medida é 2. O código não tem erro de programação:

In [ ]:
print("O mesmo código, com a posição EXATA (sem ruído):")
_, _, a_limpo = avalia_conforto(t, posicao_exata(t))
print(f"erro máximo da aceleração: {np.abs(a_limpo - aceleracao_exata(t[1:-1])).max():.1e} m/s²")
erros_sem_ruido = []
passos = [0.08, 0.04, 0.02, 0.01]
for h_ in passos:                                      # a fórmula do agente, com passos cada vez menores
    tt = np.arange(0.5, 2.5, h_)
    st = posicao_exata(tt)
    a_teste = (st[2:] - 2 * st[1:-1] + st[:-2]) / h_**2
    erros_sem_ruido.append(np.abs(a_teste - aceleracao_exata(tt[1:-1])).max())
print(f"ordem observada: {ordem_observada(passos, erros_sem_ruido):.2f}")

### 6.3 Diagnóstico

O código faz o que foi pedido, e as fórmulas são de segunda ordem. O erro está no **pedido**: "diferenças centradas de segunda ordem" com o menor passo disponível. Para dados ruidosos, a derivada segunda é um problema **mal condicionado** (capítulo 2): uma perturbação de tamanho $\sigma$ nos dados vira uma perturbação de tamanho $\sigma/h^2$ no resultado. Usar todos os pontos, com o passo mais fino, é a **pior** escolha possível.

### 6.4 A correção

Há duas saídas, e as duas trocam resolução no tempo por menos ruído.

**Passo maior.** A fórmula {eq}`eq-passo-otimo` dá o passo ótimo. Uma estimativa de $M_4 = |a''|$ sai do próprio problema: a aceleração vai de 0 a cerca de $1\ \text{m/s}^2$ e volta em uns 3 s, ou seja, $|a''| \sim (2\pi/3)^2/2 \approx 2\ \text{m/s}^4$.

**Ajuste local (Savitzky-Golay).** Em vez de passar uma parábola por três pontos, ajusta-se por **mínimos quadrados** (capítulo 7) uma parábola a uma janela de vários pontos, e deriva-se a parábola. É o filtro de Savitzky-Golay, `scipy.signal.savgol_filter(..., deriv=2)`.

Arraste o slider para mudar o passo $h = k\,\Delta t$ da diferença centrada.

In [ ]:
M4 = 2.0
h_otimo = (48 * sigma / M4) ** 0.25
print(f"passo ótimo estimado: h* = {h_otimo:.2f} s  (k = h*/Δt ≈ {h_otimo / dt:.0f} leituras)")

def acel_passo(k):
    i = np.arange(k, t.size - k)
    H = k * dt
    return t[i], (s[i + k] - 2 * s[i] + s[i - k]) / H**2

In [ ]:
def quadro_passo(k):
    tk, ak = acel_passo(k)
    erro = np.abs(ak - aceleracao_exata(tk)).max()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=tk, y=ak, mode="lines", name="diferença centrada com passo k·Δt",
                             line=dict(color=CORES["aproximacao"], width=1.5)))
    fig.add_trace(go.Scatter(x=t, y=aceleracao_exata(t), mode="lines", name="aceleração exata",
                             line=dict(color=CORES["exata"], dash="dot", width=2)))
    for y_lim in (1.2, -1.2):
        fig.add_hline(y=y_lim, line=dict(color=CORES["referencia"], dash="dash"))
    fig.update_xaxes(title="tempo (s)")
    fig.update_yaxes(title="a (m/s²)", range=[-3, 3])
    fig.update_layout(width=860, height=440, legend=dict(orientation="h", y=-0.2),
                      title=f"h = {numero(k * dt, 2)} s: máximo |a| = {numero(np.abs(ak).max(), 2)} m/s², "
                            f"erro máximo = {numero(erro, 3)} m/s²")
    return fig

ks = [1, 2, 4, 8, 16, 32]
quadros = [(numero(k * dt, 2), quadro_passo(k)) for k in ks]
mostrar(com_controle(quadros, rotulo="h (s) = ", ativo=4, titulos=[q[1].layout.title.text for q in quadros]))

Com $h = 0{,}32$ s, praticamente o passo previsto pela teoria, o erro máximo cai para cerca de $0{,}05\ \text{m/s}^2$, e a aceleração máxima medida é de $1{,}0\ \text{m/s}^2$: o elevador está **aprovado**, com folga. Com $h = 0{,}64$ s, o truncamento começa a achatar os picos. A comparação com o ajuste local confirma:

In [ ]:
from scipy.signal import savgol_filter
t_ok = (t > 0.5) & (t < 13)
_, a_passo = acel_passo(16)
for janela in [11, 31, 41]:
    a_sg = savgol_filter(s, janela, 2, deriv=2, delta=dt)
    print(f"Savitzky-Golay, janela de {janela * dt:.2f} s: máx |a| = {np.abs(a_sg[t_ok]).max():.3f} m/s², "
          f"erro máximo = {np.abs(a_sg - aceleracao_exata(t))[t_ok].max():.3f} m/s²")
print(f"diferença centrada com h = 0,32 s: máx |a| = {np.abs(a_passo).max():.3f} m/s²")

> **Lição.** "Segunda ordem" descreve o erro de truncamento, e diz nada sobre o ruído. Antes de derivar dados medidos, estime o ruído $\sigma$, preveja o erro com $\sigma/h^p$ e escolha o passo. Verifique num trecho em que a resposta é conhecida (aqui, a aceleração zero do cruzeiro) e em dados sintéticos com o mesmo ruído. Na prática, quando a grandeza importa, meça-a diretamente: um acelerômetro barato resolve o que nenhuma fórmula resolve.

## 7. Quando falha

### 7.1 Funções que não são suaves

A ordem 2 da diferença centrada vem de Taylor, que exige derivadas contínuas até a terceira. Em $f(x) = x|x|$, a segunda derivada salta em $x = 0$:

In [ ]:
f_quina = lambda x: x * np.abs(x)                     # f'(0) = 0, mas f'' salta em 0
passos = np.array([0.1, 0.05, 0.025, 0.0125])
for x_ in [0.5, 0.0]:
    e = [abs(centrada(f_quina, x_, h_) - 2 * abs(x_)) for h_ in passos]
    if max(e) < 1e-12:
        print(f"x = {x_}: erros = {[f'{v:.1e}' for v in e]}  (só arredondamento: exata para parábolas)")
    else:
        print(f"x = {x_}: erros = {[f'{v:.1e}' for v in e]};  ordem observada = {ordem_observada(passos, e):.2f}")

Longe da quina, a fórmula é exata, porque $f$ é uma parábola de cada lado. Em $x = 0$, o erro é exatamente $h$: **ordem 1**, não 2. Richardson, que supõe o erro $c_2h^2$, piora em vez de melhorar. Em dados de engenharia, quinas aparecem em contatos, válvulas que abrem, materiais que escoam. Medir a ordem observada (capítulo 2) denuncia o problema.

### 7.2 A diferenciação automática deriva o código, não a ideia

**Bisseção.** A raiz $x^*(p)$ de $x^3 + px - 1 = 0$ depende do parâmetro $p$, e pelo teorema da função implícita, $dx^*/dp = -x^*/(3x^{*2} + p)$. Calculando a raiz por bisseção e pedindo a derivada ao PyTorch:

In [ ]:
def raiz_bissecao(p, a=0.0, b=1.0, n=60):
    for _ in range(n):
        m = (a + b) / 2
        if (m**3 + p * m - 1) * (a**3 + p * a - 1) <= 0:   # só o SINAL de f é usado
            b = m
        else:
            a = m
    return torch.as_tensor((a + b) / 2)

def raiz_newton(p, x=1.0, n=30):
    x = torch.as_tensor(x)
    for _ in range(n):
        x = x - (x**3 + p * x - 1) / (3 * x**2 + p)
    return x

p = torch.tensor(2.0, requires_grad=True)
xb = raiz_bissecao(p)
print(f"bisseção: x* = {xb.item():.12f}; a saída depende de p para o PyTorch? {xb.requires_grad}")
xn = raiz_newton(p)
xn.backward()
x_ = xn.item()
print(f"Newton:   x* = {x_:.12f}; dx*/dp automática = {p.grad.item():.12f}")
print(f"teorema da função implícita: dx*/dp = {-x_ / (3 * x_**2 + 2.0):.12f}")

A bisseção usa $p$ só em **comparações**, e uma comparação tem derivada zero: para o PyTorch, a raiz não depende de $p$. A derivada automática de uma bisseção é zero (ou nem existe), embora a raiz mude com $p$. Com Newton, que usa $p$ em contas, a derivada automática converge para a correta. A lição geral: derivar um algoritmo iterativo dá a derivada **das iterações**, que pode ser diferente da derivada da solução. Para raízes e otimizações, o teorema da função implícita costuma ser o caminho certo.

**`where` com um ramo inválido.** Um código que protege uma raiz quadrada com `torch.where` dá o valor certo, mas a derivada vira `nan`:

In [ ]:
x = torch.tensor(-1.0, requires_grad=True)
y = torch.where(x > 0, torch.sqrt(x), torch.zeros(()))   # para x < 0, a saída é 0, sem problema...
y.backward()
print(f"y = {y.item()},  dy/dx = {x.grad.item()}")        # ...mas a derivada é nan
x = torch.tensor(-1.0, requires_grad=True)
y = torch.where(x > 0, torch.sqrt(torch.clamp(x, min=1e-12)), torch.zeros(()))
y.backward()
print(f"com clamp dentro do ramo: dy/dx = {x.grad.item()}")

O modo reverso calcula a derivada **dos dois ramos** e multiplica a do ramo descartado por zero. Mas a derivada de $\sqrt{x}$ em $x = -1$ é `nan`, e $0 \times \text{nan} = \text{nan}$. A correção é garantir que o ramo descartado também seja válido.

### 7.3 Precisão simples

O PyTorch usa `float32` por padrão, com $\epsilon \approx 1{,}2 \times 10^{-7}$. Para redes neurais, basta. Para diferenças finitas, o fundo do V da seção 5.1 sobe muito:

In [ ]:
for tipo in [torch.float32, torch.float64]:
    xt = torch.tensor(1.0, dtype=tipo)
    melhor = min(abs(((torch.sin(xt + h_) - torch.sin(xt - h_)) / (2 * h_)).item() - np.cos(1.0))
                 for h_ in np.logspace(-8, -1, 29))
    print(f"{str(tipo):14s}: melhor erro da diferença centrada = {melhor:.1e}")

Em `float32`, a diferença centrada nunca passa de seis algarismos corretos, contra doze em `float64`. Ao verificar um gradiente automático com diferenças finitas (como `torch.autograd.gradcheck` faz), use `float64`.

## Resumo

| Método | Fórmula | Erro | Quando usar |
| --- | --- | --- | --- |
| Diferença progressiva | $[f(x+h) - f(x)]/h$ | $O(h)$; ótimo $h \sim 10^{-8}$ | pontas de tabelas, quando nada melhor cabe |
| Diferença centrada | $[f(x+h) - f(x-h)]/2h$ | $O(h^2)$; ótimo $h \sim 10^{-5}$ | padrão para funções e dados lisos |
| Segunda derivada centrada | $[f(x+h) - 2f(x) + f(x-h)]/h^2$ | $O(h^2)$; ruído $\times 1/h^2$ | funções; dados só com passo escolhido |
| Richardson | $[4D(h/2) - D(h)]/3$ | $O(h^4)$ | funções suaves, sem ruído |
| Passo complexo | $\operatorname{Im} f(x+ih)/h$ | precisão de máquina | código analítico que aceita complexos |
| Savitzky-Golay | mínimos quadrados locais | equilibra truncamento e ruído | dados medidos |
| Diferenciação automática | regra da cadeia no programa | só arredondamento | gradientes de programas, otimização, redes neurais |

## Exercícios

**Conceituais (sem computador)**

1. Deduza a fórmula unilateral de três pontos $f'(x_0) \approx [-3f(x_0) + 4f(x_0+h) - f(x_0+2h)]/(2h)$ e o seu erro.
2. Mostre que o passo ótimo da diferença centrada é $h^* = (3\epsilon|f|/|f'''|)^{1/3}$ e calcule o erro mínimo.
3. Para a segunda derivada com ruído independente de desvio padrão $\sigma$ em cada leitura, mostre que o desvio padrão do resultado é $\sqrt6\,\sigma/h^2$.
4. Faça à mão, com números duais, a derivada de $f(x) = x^2 \sin x$ em $x = \pi$.

**Verificação (com agente)**

5. Peça a um agente o **jerk** (derivada da aceleração) do elevador, com limite de $2\ \text{m/s}^3$. Preveja o fator de amplificação do ruído para a terceira derivada, escolha o passo e verifique no trecho de velocidade constante.
6. Peça a um agente uma função que calcule o gradiente de $f(\mathbf{x}) = \sum_i (1 - x_i)^2 + 100(x_{i+1} - x_i^2)^2$ para $n = 1000$ por diferenças finitas, e compare tempo e precisão com `torch.autograd`. Quantas avaliações de $f$ cada um usa?
7. Peça a um agente a derivada de uma função tabelada com espaçamento irregular (por exemplo, a curva de viscosidade do capítulo 6). Verifique se o código trata as pontas e o espaçamento irregular, comparando com `np.gradient(y, x, edge_order=2)` e com um polinômio de grau 2 conhecido.

**Projeto aberto**

8. **Velocidade a partir de um GPS.** Um GPS registra a posição de um veículo a 1 Hz, com erro de 3 m. Estime a velocidade e a aceleração, compare passo maior, Savitzky-Golay e um ajuste por *spline* com suavização (`scipy.interpolate.make_smoothing_spline`). Qual é a menor duração de frenagem que você consegue detectar?

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulo 23.
- A. Griewank e A. Walther, *Evaluating Derivatives: Principles and Techniques of Algorithmic Differentiation*, 2ª ed., SIAM, 2008.
- J. R. R. A. Martins, P. Sturdza e J. J. Alonso, "The complex-step derivative approximation", *ACM Transactions on Mathematical Software*, 29(3), 2003.
- A. Savitzky e M. J. E. Golay, "Smoothing and differentiation of data by simplified least squares procedures", *Analytical Chemistry*, 36(8), 1964.